# 春日野穹 GPT‑SoVITS 全量微调（GPT句内情绪条件层 + GPT预测主干 + SoVITS 2,575条语音音色）

把 `sora_condition_colab.zip` 放到 Google Drive 的 `MyDrive` 根目录。选 GPU，依次运行下面两个单元。支持断点续训。

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, os, subprocess, sys, urllib.request, zipfile

drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive')
sora_dir = drive_root / '春日野穹_GPTSoVITS_训练'
bundle = sora_dir / 'sora_condition_colab.zip'
assert bundle.is_file(), f'找不到 {bundle}'
job = Path('/content/sora_condition')
job.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(job)

repo = job / 'GPT-SoVITS'
if not repo.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout',
                    'https://github.com/RVC-Boss/GPT-SoVITS.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach',
                '1b47ca07ef43a284e0b8bd7f29d3e1e3ff76082a'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torchmetrics==1.5.0', 'tqdm', 'x-transformers'], check=True)
subprocess.run(['nvidia-smi'], check=True)

HF_BASE = 'https://huggingface.co/lj1995/GPT-SoVITS/resolve/main'
WEIGHTS = {
    's1v3.ckpt': (
        f'{HF_BASE}/s1v3.ckpt',
        '87133414860ea14ff6620c483a3db5ed07b44be42e2c3fcdad65523a729a745a',
    ),
    's2Gv2ProPlus.pth': (f'{HF_BASE}/v2Pro/s2Gv2ProPlus.pth', None),
    's2Dv2ProPlus.pth': (f'{HF_BASE}/v2Pro/s2Dv2ProPlus.pth', None),
}

def sha256_file(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

for fname, (url, expected_sha) in WEIGHTS.items():
    target = job / fname
    if not target.is_file() or (expected_sha and sha256_file(target) != expected_sha):
        print(f'正在下载预训练底座 {fname} ...', flush=True)
        pending = target.with_suffix('.pending')
        urllib.request.urlretrieve(url, pending)
        if expected_sha:
            assert sha256_file(pending) == expected_sha, f'{fname} SHA256 校验失败'
        pending.replace(target)
print('环境与全部 2,575 条语音训练数据准备完毕。', flush=True)


In [ ]:
env = dict(os.environ, SORAVOICE_APP=str(repo), PYTHONPATH=f"{repo / 'GPT_SoVITS'}:{repo}")
out_dir = sora_dir / 'sora_condition_results_v2proplus'

# 1. 训练 GPT（s1v3 底座 + 句内情绪条件层 Adapter + GPT 语义预测主干联合微调）
gpt_cmd = [sys.executable, '-u', str(job / 'train_condition.py'),
           '--data', str(job / 'training-data.pt'),
           '--base', str(job / 's1v3.ckpt'),
           '--batch-size', '8',
           '--out', str(out_dir)]
subprocess.run(gpt_cmd + ['--smoke'], env=env, cwd=job, check=True)
subprocess.run(gpt_cmd + ['--epochs', '15'], env=env, cwd=job, check=True)

# 2. 训练 SoVITS v2ProPlus（2,575 条春日野穹真实语音 + HuBERT + SV声纹 微调）
sovits_cmd = [sys.executable, '-u', str(job / 'train_sovits.py'),
              '--data', str(job / 'training-data.pt'),
              '--prep', str(job / 'prep'),
              '--base-g', str(job / 's2Gv2ProPlus.pth'),
              '--base-d', str(job / 's2Dv2ProPlus.pth'),
              '--batch-size', '8',
              '--out', str(out_dir)]
subprocess.run(sovits_cmd + ['--smoke'], env=env, cwd=job, check=True)
subprocess.run(sovits_cmd + ['--epochs', '8'], env=env, cwd=job, check=True)

# 3. 校验产物并同步释放 Colab GPU
import io, sys, torch
from google.colab import runtime

if str(repo / 'GPT_SoVITS') not in sys.path:
    sys.path.insert(0, str(repo / 'GPT_SoVITS'))

weights = torch.load(out_dir / 'phrase-adapter-epoch15.pt', map_location='cpu', weights_only=True)
gpt_ckpt = torch.load(out_dir / 'sora-gpt-v2-epoch15.ckpt', map_location='cpu', weights_only=True)
sovits_bytes = (out_dir / 'sora-sovits-latest.pth').read_bytes()
if sovits_bytes[:2] == b'06':
    sovits_bytes = b'PK' + sovits_bytes[2:]
sovits_ckpt = torch.load(io.BytesIO(sovits_bytes), map_location='cpu', weights_only=False)
assert weights.get('adapter') and weights.get('gpt_weight'), 'GPT+Adapter 权重缺失'
assert gpt_ckpt.get('weight') and sovits_ckpt.get('weight'), 'GPT 或 SoVITS 导出权重缺失'
print('v2ProPlus 全套权重（s1v3+Adapter 与 SoVITS v2ProPlus）已全部验证通过，正在同步 Drive。', flush=True)
drive.flush_and_unmount()
print('Drive 已同步，正在释放 GPU 运行时。', flush=True)
runtime.unassign()
